# tegkit in 5 minutes

**How much power can a thermoelectric generator actually make?** This notebook answers that with
[tegkit](https://github.com/aysel-elsh/tegkit), using a real module's datasheet numbers.

Run each cell with **Shift + Enter**. Nothing to install on your computer.

In [ ]:
%pip install -q tegkit matplotlib

## 1. Pick a real module

tegkit ships presets built from public datasheets. Each one records the datasheet it came from.

In [ ]:
import tegkit
from tegkit import presets, iv_curve, max_power_point, suggest_array, c_to_k
import matplotlib.pyplot as plt

print("Presets:", presets.names())
info = presets.info("TG12-4")
print(info.manufacturer, "|", info.conditions)
print("Datasheet:", info.source)
m = info.module
m

## 2. Power vs. load at a given ΔT

A TEG acts like a voltage source (Seebeck voltage) with an internal resistance. Power peaks when
the load matches that resistance.

In [ ]:
DT = 100  # kelvin across the module faces

pts = iv_curve(m, DT, points=41)
mpp = max_power_point(m, DT)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot([p.current for p in pts], [p.voltage for p in pts], label="Voltage")
ax1.set_xlabel("Current (A)"); ax1.set_ylabel("Voltage (V)")
ax2 = ax1.twinx()
ax2.plot([p.current for p in pts], [p.power for p in pts], color="tab:orange", label="Power")
ax2.plot(mpp.current, mpp.power, "o", color="tab:red")
ax2.set_ylabel("Power (W)")
ax1.set_title(f"{m.name} at ΔT = {DT} K: max {mpp.power:.2f} W at {mpp.voltage:.2f} V")
fig.tight_layout(); plt.show()

## 3. The heat-sink reality check

The ΔT your heat source offers is **not** the ΔT the module sees. Thermal resistance in the hot-side
and cold-side heat sinks eats part of it. This catches out most first designs.

In [ ]:
source_dt = 150  # K between the heat source and ambient air
sinks = [0.05, 0.1, 0.2, 0.4, 0.8, 1.6]  # cold-side heat-sink resistance, K/W

rows = []
for r_cold in sinks:
    dt = m.module_delta_t(source_delta_t=source_dt, hot_side_resistance=0.1, cold_side_resistance=r_cold)
    rows.append((r_cold, dt, m.max_power(dt)))
    print(f"heat sink {r_cold:>4} K/W -> module sees {dt:6.1f} K -> {m.max_power(dt):5.2f} W")

plt.figure(figsize=(6, 3.5))
plt.plot([r[0] for r in rows], [r[2] for r in rows], "o-")
plt.xscale("log"); plt.xlabel("Cold-side heat sink (K/W)"); plt.ylabel("Max power (W)")
plt.title("A better heat sink is often worth more than a better module"); plt.grid(alpha=.3); plt.show()

## 4. Size an array

Say you need about **20 W at 12 V**. tegkit picks a series × parallel layout.

In [ ]:
arr = suggest_array(m, delta_t=DT, target_power=20, target_voltage=12)
print(f"{arr.series} in series x {arr.parallel} in parallel = {arr.count} modules")
print(f"Open-circuit voltage: {arr.open_circuit_voltage(DT):.1f} V")
print(f"Matched load: {arr.matched_load():.2f} ohm -> {arr.max_power(DT):.1f} W")

## 5. Efficiency limit

Efficiency is low for TEGs, usually a few percent. Here's the ideal maximum for this module over a
range of hot-side temperatures, with the cold side at 25 °C.

In [ ]:
hot = list(range(50, 251, 10))
eff = [m.max_efficiency(c_to_k(t), c_to_k(25)) * 100 for t in hot]
plt.figure(figsize=(6, 3.5))
plt.plot(hot, eff)
plt.xlabel("Hot side (°C)"); plt.ylabel("Max efficiency (%)"); plt.grid(alpha=.3)
plt.title(f"{m.name}: ideal max efficiency, cold side 25 °C"); plt.show()

## 6. Your turn

Change the numbers below to model your own setup, or build a module from your own datasheet with
`TEGModule.from_datasheet(...)` or bench measurements with `TEGModule.from_measurements(...)`.

In [ ]:
MODULE = "TG12-4"     # any name from presets.names()
SOURCE_DT = 120        # K available from your heat source
HOT_SINK, COLD_SINK = 0.1, 0.3   # K/W
SERIES, PARALLEL = 2, 2

mod = presets.get(MODULE)
dt = mod.module_delta_t(SOURCE_DT, HOT_SINK, COLD_SINK)
a = tegkit.TEGArray(mod, series=SERIES, parallel=PARALLEL)
print(f"Module ΔT: {dt:.1f} K | array Voc: {a.open_circuit_voltage(dt):.2f} V | "
      f"max power: {a.max_power(dt):.2f} W into {a.matched_load():.2f} ohm")

---
Found a bug or want a new feature? [Open an issue](https://github.com/aysel-elsh/tegkit/issues).
Adding a datasheet preset is a great first PR.